[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/38_grpo_loss.ipynb)

# 🔴 Hard: GRPO Loss

Implement the **Group Relative Policy Optimization (GRPO)** loss — a group-wise, baseline-subtracted REINFORCE objective commonly used in RLAIF (reinforcement learning from AI feedback).

Given a batch of log-probabilities, scalar rewards, and group ids (one group per prompt), define the within-group normalized advantages:

$$A_i = \frac{r_i - \bar r_{g(i)}}{\text{std}_{g(i)} + \epsilon}$$

where \(\bar r_{g(i)}\) and \(\text{std}_{g(i)}\) are the mean and standard deviation of rewards in the group of example \(i\).

The GRPO loss is then the negative advantage-weighted log-probability:

$$\mathcal{L}_{\text{GRPO}} = -\mathbb{E}_i \big[\,\text{stop\_grad}(A_i)\, \log \pi_\theta(y_i)\big].$$

### Signature
```python
from torch import Tensor

def grpo_loss(logps: Tensor, rewards: Tensor, group_ids: Tensor,
              eps: float = 1e-5) -> Tensor:
    """GRPO loss over a batch.

    logps: (B,) policy log-probs for each sampled response
    rewards: (B,) scalar rewards for each response
    group_ids: (B,) integers, same id = same prompt/group
    returns: scalar loss (Tensor)
    """
```

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.5 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn.functional as F

In [5]:
# ✏️ YOUR IMPLEMENTATION HERE

from torch import Tensor

def grpo_loss_vectorized(logps: Tensor, rewards: Tensor, group_ids: Tensor,
              eps: float = 1e-5) -> Tensor:
    # rewards: (n,)
    # logps: (n,)
    # group_ids: (n,)

    num_groups = group_ids.max() + 1
    # rewards mean by group
    mean_per_group = torch.scatter_reduce(torch.zeros((num_groups,)), dim=0, index=group_ids, src=rewards, reduce="mean", include_self=False)
    mean_rewards = mean_per_group[group_ids] # (n,)

    # reward std by group, std = mean_over_group((x - mean)^2)
    mean_diff_square = (rewards - mean_rewards) ** 2

    var_per_group = torch.scatter_reduce(torch.zeros((num_groups,)), dim=0, index=group_ids, src=mean_diff_square, reduce="mean", include_self=False) # (num_groups,)
    std_per_group = torch.sqrt(var_per_group)

    std_rewards = std_per_group[group_ids] # (n,)

    Adv = (rewards - mean_rewards) / (std_rewards + eps)

    return -torch.mean(Adv.detach() * logps) # Note: detach because we don't want any gradient to flow through adv, only log_prob


def grpo_loss(logps: Tensor, rewards: Tensor, group_ids: Tensor,
              eps: float = 1e-5) -> Tensor:
    # rewards: (n,)
    # logps: (n,)
    # group_ids: (n,)

    num_groups = group_ids.max() + 1
    group_mean_reward = torch.zeros((num_groups,))
    group_std_reward = torch.zeros((num_groups,))

    for group_id in range(num_groups):
      # find rewards belong to this group
      valid_group_idx = group_ids == group_id # (n,)
      group_rewards = rewards[valid_group_idx]
      group_mean_reward[group_id] = torch.mean(group_rewards, dim=0)
      group_std_reward[group_id] = torch.std(group_rewards, dim=0, unbiased=False)

    mean_rewards = group_mean_reward[group_ids]
    std_rewards = group_std_reward[group_ids]

    Adv = (rewards - mean_rewards) / (std_rewards + eps)

    return -torch.mean(Adv.detach() * logps) # Note: detach because we don't want any gradient to flow through adv, only log_prob





In [6]:
rewards = torch.tensor([1.0, 0.8, 0.2, 0.0])
group_ids = torch.tensor([0, 0, 1, 1])
num_groups = group_ids.max() + 1

group_id = 0
valid_group_idx = group_ids == group_id
rewards[valid_group_idx]

# mean_per_group = torch.scatter_reduce(torch.zeros((num_groups,)), dim=0, index=group_ids, src=rewards, reduce="mean", include_self=False)
# mean_rewards = mean_per_group[group_ids]
# mean_rewards

tensor([1.0000, 0.8000])

In [7]:
# 🧪 Debug
logps = torch.tensor([0.0, -0.5, -1.0, -1.5])
rewards = torch.tensor([1.0, 0.8, 0.2, 0.0])
group_ids = torch.tensor([0, 0, 1, 1])
print('Loss:', grpo_loss(logps, rewards, group_ids).item())

Loss: -0.24997496604919434


In [8]:
# ✅ SUBMIT
from torch_judge import check
check('grpo_loss')


🧪 Testing: GRPO (Group Relative Policy Optimization) Loss (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Basic shape & type (7.1ms)
  ✅ [2/4] Numeric check vs reference (24.7ms)
  ✅ [3/4] Gradient flows to logps only (6.6ms)
  ✅ [4/4] Group-wise normalization (9.5ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (47.9ms total)
  Progress saved. Run status() to see your dashboard.

